# 🌲 Wildfire ResNet18 Training
## **5-Day Vegetation-Only Features**

This notebook executes a complete training and testing pipeline based on the reference paper's methodology.
**Key elements include:**
- ImageNet pre-trained weights for improved feature extraction
- Focal Loss to address extreme class imbalance
- `val_AP` monitoring during training
- Automated model export to ONNX format for orbital satellite deployment


### 1. Setup Environment
Install the necessary packages, mount Google Drive to persist data and checkpoints, and clone the target repository.


In [1]:
!pip install -q pytorch-lightning wandb segmentation-models-pytorch h5py scikit-learn tqdm pyyaml rasterio onnx onnxruntime
!pip install -U "jsonargparse[signatures]>=4.27.7"

import os
import sys
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone repository
!cd /content && git clone https://github.com/SebastianGer/WildfireSpreadTS.git
os.chdir('/content/WildfireSpreadTS')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.2/852.2 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 96.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.6/787.6 kB 17.0 MB/s eta 0:00:00
Mounted at /content/drive
Cloning into 'WildfireSpreadTS'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 195 (delta 70), reused 54 (delta 54), pack-reused 97 (from 1)
Receiving objects: 100% (195/195), 64.87 KiB | 4.99 MiB/s, done.
Resolving deltas: 100% (109/109), done.


In [2]:
# 🔴 PATCH 1: Correzione del Bug di ereditarietà spuria in SMPModel.py
with open("src/models/SMPModel.py", "w") as f:
    f.write("""from typing import Any
import segmentation_models_pytorch as smp
from .BaseModel import BaseModel

class SMPModel(BaseModel):
    def __init__(
        self,
        encoder_name: str,
        n_channels: int,
        flatten_temporal_dimension: bool,
        pos_class_weight: float,
        loss_function: str,  # 🟢 ECCCOLA QUI: Dichiarata esplicitamente per il validatore!
        *args: Any,
        **kwargs: Any
    ):
        super().__init__(
            n_channels=n_channels,
            flatten_temporal_dimension=flatten_temporal_dimension,
            pos_class_weight=pos_class_weight,
            loss_function=loss_function,  # 🟢 Passata in modo trasparente al padre!
            *args,
            **kwargs
        )
        self.save_hyperparameters()

        self.model = smp.Unet(
            encoder_name=encoder_name,
            encoder_weights="imagenet",
            in_channels=n_channels,
            classes=1,
        )
""")

# 🟢 PATCH 2: Risoluzione dell'ImportError 'T_co' per compatibilità Python 3.12 / PyTorch 2.x
# Leggiamo il file originale e sostituiamo l'import obsoleto con la sintassi universale protetta
dataset_path = "src/dataloader/FireSpreadDataset.py"
with open(dataset_path, "r") as f:
    dataset_code = f.read()

# Sostituiamo il vecchio import di T_co con un fallback sicuro che non genera crash
dataset_code = dataset_code.replace(
    "from torch.utils.data.dataset import T_co",
    "from typing import TypeVar\nT_co = TypeVar('T_co', covariant=True)"
)

with open(dataset_path, "w") as f:
    f.write(dataset_code)

# 🔵 PATCH 3: Risveglio forzato di Wandb (Risolve l'AttributeError NoneType su wandb.run.dir)
# Forziamo l'accesso a cli.trainer.logger.experiment prima di wandb_setup per svegliare la sessione
train_path = "src/train.py"
with open(train_path, "r") as f:
    train_code = f.read()

train_code = train_code.replace(
    "cli.wandb_setup()",
    "if cli.trainer and cli.trainer.logger: _ = cli.trainer.logger.experiment\n    cli.wandb_setup()"
)

with open(train_path, "w") as f:
    f.write(train_code)

# 🟣 PATCH 4: Correzione del calcolo di Alpha per la Focal Loss in BaseModel.py
base_model_path = "src/models/BaseModel.py"
with open(base_model_path, "r") as f:
    base_model_code = f.read()

# Sostituzione mirata a riga singola (immune a variazioni di indentazione)
base_model_code = base_model_code.replace(
    "alpha=1 - self.hparams.pos_class_weight",
    "alpha=self.hparams.pos_class_weight / (1 + self.hparams.pos_class_weight)"
)
base_model_code = base_model_code.replace(
    "alpha=1-self.hparams.pos_class_weight",
    "alpha=self.hparams.pos_class_weight / (1 + self.hparams.pos_class_weight)"
)

with open(base_model_path, "w") as f:
    f.write(base_model_code)

print("✅ File src/models/SMPModel.py patchato (ImageNet Hardcoded)!")
print("✅ File src/dataloader/FireSpreadDataset.py patchato (T_co Python 3.12 Fix)!")
print("✅ File src/train.py patchato (Forced Wandb Init Fix)!")
print("✅ File src/models/BaseModel.py patchato (Focal Loss Alpha Fix)!")

✅ File src/models/SMPModel.py patchato (ImageNet Hardcoded)!
✅ File src/dataloader/FireSpreadDataset.py patchato (T_co Python 3.12 Fix)!
✅ File src/train.py patchato (Forced Wandb Init Fix)!
✅ File src/models/BaseModel.py patchato (Focal Loss Alpha Fix)!


### 2. Weights & Biases Authentication
We use Weights & Biases for experiment tracking. Login to store logs, metrics, and models.


In [3]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: guazzi-ale (guazzi-ale-politecnico-di-milano) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### 3. Setup Paths and Validate Data
Configure our primary data directory on Google Drive and output directories. We also verify the amount of available HDF5 data files per year.


In [4]:
DATA_DIR = "/content/drive/MyDrive/Wildfire_Project"
OUTPUT_DIR = "/content/lightning_logs"
CONFIGS_DIR = "/content/WildfireSpreadTS/cfgs"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
for year in [2018, 2019, 2020, 2021]:
    year_dir = Path(DATA_DIR) / str(year)
    if year_dir.exists():
        hdf5_files = list(year_dir.glob("*.hdf5"))
        print(f"  {year}: {len(hdf5_files)} HDF5 files")

total_hdf5 = len(list(Path(DATA_DIR).glob("*/*.hdf5")))
print(f"\nTotal HDF5 files: {total_hdf5}")

Data directory: /content/drive/MyDrive/Wildfire_Project
  2018: 176 HDF5 files
  2019: 74 HDF5 files
  2020: 201 HDF5 files
  2021: 156 HDF5 files

Total HDF5 files: 607


### 4. Create Data Configuration (Vegetation-Only)
This configuration applies a **5-day history observation** with a batch size of 64.
We are retaining only vegetation and active fire features: `VIIRS (0-2) + NDVI (3) + EVI2 (4) + Active Fire (38-39)`.


In [10]:
data_config = f"""
data_dir: {DATA_DIR}
batch_size: 64
n_leading_observations: 5
crop_side_length: 128
load_from_hdf5: true
num_workers: 2  # 🟢 FIX: Impostato a 2 per evitare il freeze di Google Drive a inizio epoca
remove_duplicate_features: true
features_to_keep: [0, 1, 2, 3, 4, 38, 39]
n_leading_observations_test_adjustment: 5
"""

with open(f"{CONFIGS_DIR}/data_colab_paper.yaml", "w") as f:
    f.write(data_config)

print("✓ Data config created: data_colab_paper.yaml")
print("  - Features: Vegetation + Active Fire only")
print("  - Batch size: 64 (come nel paper)")
print("  - n_leading_observations: 5 giorni")

✓ Data config created: data_colab_paper.yaml
  - Features: Vegetation + Active Fire only
  - Batch size: 64 (come nel paper)
  - n_leading_observations: 5 giorni


### 5. Create Trainer Configuration
Setting up the PyTorch Lightning trainer.
**Important Fix:** We monitor `val_AP` (Average Precision calculated at the end of each validation epoch) instead of `test_AP` (which is only available after training).


In [11]:
# 🔴 FIX: Usa val_AP invece di test_AP!
# test_AP esiste SOLO durante la fase di test (Cella 11)
# Durante il training, PyTorch Lightning calcola val_AP alla fine di ogni epoca

trainer_config = """
accelerator: gpu
strategy: auto
devices: 1
num_nodes: 1
precision: 32-true
logger:
  class_path: pytorch_lightning.loggers.wandb.WandbLogger
  init_args:
    project: wildfire_resnet18_paper_based
    log_model: true
callbacks:
  - class_path: pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint
    init_args:
      monitor: val_AP
      mode: max
max_steps: 10000
check_val_every_n_epoch: 1
enable_progress_bar: true
default_root_dir: """ + OUTPUT_DIR

with open(f"{CONFIGS_DIR}/trainer_colab_paper.yaml", "w") as f:
    f.write(trainer_config)

print("✓ Trainer config created: trainer_colab_paper.yaml")
print("  - Checkpoint monitor: val_AP (CORRETTO - durante training)")
print("  - Max steps: 10,000 (esattamente come nel paper)")
print("\n  🔴 NOTA IMPORTANTE:")
print("     - val_AP: metrica DURANTE il training (ogni epoca)")
print("     - test_AP: metrica DOPO il training (fase test finale)")

✓ Trainer config created: trainer_colab_paper.yaml
  - Checkpoint monitor: val_AP (CORRETTO - durante training)
  - Max steps: 10,000 (esattamente come nel paper)

  🔴 NOTA IMPORTANTE:
     - val_AP: metrica DURANTE il training (ogni epoca)
     - test_AP: metrica DOPO il training (fase test finale)


### 6. Verify HDF5 Structure and Dynamic Channel Calculation
Before setting up the model, we instantiate our `FireSpreadDataModule` to determine the exact number of input channels resulting from the temporal stacking of the selected features.
*(Split into two cells to isolate the dataloader instantiation and the actual data extraction).*


In [12]:
# 🟢 HOTFIX PER COMPATIBILITÀ PYTORCH (Risolve l'errore T_co)
import torch.utils.data.dataset
from typing import TypeVar
torch.utils.data.dataset.T_co = TypeVar('T_co', covariant=True)

import h5py
import torch
from src.dataloader.FireSpreadDataModule import FireSpreadDataModule
from src.dataloader.FireSpreadDataset import FireSpreadDataset

print("\n" + "="*70)
print("FASE 1: CALCULATE CHANNELS WITH VEGETATION-ONLY FEATURES")
print("="*70)

print("\n1️⃣ Verifying HDF5 structure...\n")

hdf5_files = list(Path(DATA_DIR).glob("*/*.hdf5"))
if not hdf5_files:
    raise FileNotFoundError(f"No HDF5 files found in {DATA_DIR}")

test_file = hdf5_files[0]
print(f"   Test file: {test_file.name}")

with h5py.File(str(test_file), 'r') as f:
    data = f['data']
    hdf5_shape = data.shape
    print(f"   - Shape: {hdf5_shape}")
    print(f"   - Images per fire: {hdf5_shape[0]}")
    print(f"   - Channels per image: {hdf5_shape[1]}")
    print(f"   - Spatial size: {hdf5_shape[2]}×{hdf5_shape[3]}")


FASE 1: CALCULATE CHANNELS WITH VEGETATION-ONLY FEATURES

1️⃣ Verifying HDF5 structure...

   Test file: fire_23654679.hdf5
   - Shape: (28, 23, 336, 293)
   - Images per fire: 28
   - Channels per image: 23
   - Spatial size: 336×293


In [13]:
print("\n2️⃣ Loading datamodule with vegetation-only features...\n")

datamodule = FireSpreadDataModule(
    data_dir=DATA_DIR,
    batch_size=8,
    n_leading_observations=5,
    n_leading_observations_test_adjustment=5,
    crop_side_length=128,
    load_from_hdf5=True,
    num_workers=0,
    remove_duplicate_features=True,
    features_to_keep=[0, 1, 2, 3, 4, 38, 39]
)

datamodule.setup("fit")
train_loader = datamodule.train_dataloader()

# Extract a sample to dynamically determine channels
sample_batch = next(iter(train_loader))
x_sample, y_sample = sample_batch

print("   ✓ Datamodule loaded successfully")
print(f"   - Input shape: {x_sample.shape}")
print(f"   - Label shape: {y_sample.shape}")

actual_n_channels = x_sample.shape[1]

print("\n3️⃣ Channel calculation result:\n")
print(f"   ✅ ACTUAL CHANNELS: {actual_n_channels}")
print("   Features: VIIRS (0-2) + NDVI (3) + EVI2 (4) + Active Fire (38-39)")
print("   5 giorni × 7 features = 35 base channels")

print("\n4️⃣ Dataset statistics:\n")
print(f"   - Training samples: {len(datamodule.train_dataset)}")
print(f"   - Validation samples: {len(datamodule.val_dataset)}")
print("\n" + "="*70)


2️⃣ Loading datamodule with vegetation-only features...

Using the following dataset split:
Train years: [2018, 2019], Val years: [2020], Test years: [2021]
   ✓ Datamodule loaded successfully
   - Input shape: torch.Size([8, 35, 128, 128])
   - Label shape: torch.Size([8, 128, 128])

3️⃣ Channel calculation result:

   ✅ ACTUAL CHANNELS: 35
   Features: VIIRS (0-2) + NDVI (3) + EVI2 (4) + Active Fire (38-39)
   5 giorni × 7 features = 35 base channels

4️⃣ Dataset statistics:

   - Training samples: 3948
   - Validation samples: 3287



### 7. Create Model Configuration
Define a `res18_paper_based.yaml` using:
- **ResNet18 Encoder** pretrained on ImageNet
- **Focal Loss** to penalize "easy" pixels and improve the network's focus on the rare fire pixels (due to massive class imbalance).


In [14]:
print("\n" + "="*70)
print("FASE 2: CREATE MODEL CONFIG - PAPER-BASED CONFIGURATION")
print("="*70 + "\n")

model_config = f"""
seed_everything: 0
optimizer:
  class_path: torch.optim.AdamW
  init_args:
    lr: 0.001
model:
  class_path: models.SMPModel
  init_args:
    encoder_name: resnet18
    n_channels: {actual_n_channels}
    flatten_temporal_dimension: true
    pos_class_weight: 236
    loss_function: Focal
do_train: true
do_predict: false
do_test: false
"""

model_config_path = f"{CONFIGS_DIR}/unet/res18_paper_based.yaml"

with open(model_config_path, "w") as f:
    f.write(model_config)

print(f"✓ Model config created: res18_paper_based.yaml")
print(f"  Location: {model_config_path}")
print(f"  n_channels: {actual_n_channels}")
print("\n  🔴 PAPER-BASED CHANGES:")
print("     ✓ encoder_weights: 'imagenet' (pre-trained)")
print("     ✓ loss_function: 'Focal' (per class imbalance)")
print("     ✓ Focal Loss: penalizza i pixel facili, focus sui pixel rari di fuoco")


FASE 2: CREATE MODEL CONFIG - PAPER-BASED CONFIGURATION

✓ Model config created: res18_paper_based.yaml
  Location: /content/WildfireSpreadTS/cfgs/unet/res18_paper_based.yaml
  n_channels: 35

  🔴 PAPER-BASED CHANGES:
     ✓ encoder_weights: 'imagenet' (pre-trained)
     ✓ loss_function: 'Focal' (per class imbalance)
     ✓ Focal Loss: penalizza i pixel facili, focus sui pixel rari di fuoco


### 8. Verify Model Configuration (Dry Run)
Before starting a long training job, let's verify that the model architecture can be instantiated and accepts input tensors of the correct shape.


In [15]:
from src.models.SMPModel import SMPModel

print("\n" + "="*70)
print("FASE 3: VERIFY MODEL WITH IMAGENET WEIGHTS (DRY RUN)")
print("="*70 + "\n")

print("Creating ResNet18 with ImageNet pre-trained weights...\n")

# 🔴 FIX: Passa esplicitamente encoder_weights="imagenet" nel dry-run
# Questo testa che SMP adatta i canali d'ingresso correttamente
model = SMPModel(
    encoder_name="resnet18",
    n_channels=actual_n_channels,
    flatten_temporal_dimension=True,
    pos_class_weight=236,
    loss_function="Focal",
    #encoder_weights="imagenet"   ← ESPLICITO per coerenza accademica
    # Non più Esplicito perché ho cambiato il codice dando come default imagenet, o mi da errore
)

print("✓ Model created successfully!")
print("  - Architecture: U-Net with ResNet18 encoder")
print("  - Encoder weights: ImageNet (pre-trained)")
print(f"  - Input channels: {actual_n_channels} (vegetation-only)")
print("  - Output channels: 1 (binary segmentation)")
print("  - Loss: Focal Loss (for class imbalance)")

print("\nTesting forward pass...\n")

with torch.no_grad():
    x_test = torch.randn(2, actual_n_channels, 128, 128)
    y_test = model(x_test)
    print("✓ Forward pass successful!")
    print(f"  - Input shape: {x_test.shape}")
    print(f"  - Output shape: {y_test.shape}")
    print(f"  - Encoder correctly adapted to {actual_n_channels} input channels")

print("\n" + "="*70)


FASE 3: VERIFY MODEL WITH IMAGENET WEIGHTS (DRY RUN)

Creating ResNet18 with ImageNet pre-trained weights...



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

✓ Model created successfully!
  - Architecture: U-Net with ResNet18 encoder
  - Encoder weights: ImageNet (pre-trained)
  - Input channels: 35 (vegetation-only)
  - Output channels: 1 (binary segmentation)
  - Loss: Focal Loss (for class imbalance)

Testing forward pass...

✓ Forward pass successful!
  - Input shape: torch.Size([2, 35, 128, 128])
  - Output shape: torch.Size([2, 1, 128, 128])
  - Encoder correctly adapted to 35 input channels



### 9. Main Training Loop
We use standard `python src/train.py` invoking PyTorch Lightning CLI.


In [16]:
print("\n" + "="*80)
print("FASE 4: START TRAINING - PAPER-BASED CONFIGURATION")
print("="*80)

print("\n📋 Training configuration summary:")
print("   Model config: res18_paper_based.yaml")
print("   Data config: data_colab_paper.yaml (vegetation-only)")
print("   Trainer config: trainer_colab_paper.yaml")
print("\n   🔴 PAPER-BASED MODIFICATIONS:")
print("   ✓ Encoder weights: ImageNet (better feature recognition)")
print("   ✓ Loss function: Focal Loss (handles class imbalance)")
print("   ✓ Features: Vegetation only (VIIRS + NDVI + EVI2 + Fire)")
print("   ✓ Batch size: 64 (exactly as in paper)")
print("   ✓ Max steps: 10,000 (exactly as in paper)")
print("   ✓ Checkpoint monitor: val_AP (DURANTE training)")
print(f"   ✓ Training samples: {len(datamodule.train_dataset)}")
print(f"   ✓ Validation samples: {len(datamodule.val_dataset)}\n")

print("⚡ TRAINING PHASES:")
print("   1. Training phase: aggiorna pesi basato su train loss")
print("   2. Validation phase (ogni epoca): calcola val_AP, salva checkpoint se migliore")
print("   3. Test phase (dopo training): calcola test_AP come metrica finale\n")

training_command = f"""
cd /content/WildfireSpreadTS && \
python src/train.py \
  -c cfgs/unet/res18_paper_based.yaml \
  --trainer cfgs/trainer_colab_paper.yaml \
  --data cfgs/data_colab_paper.yaml \
  --do_train true \
  --do_test false \
  --do_validate false \
  --trainer.default_root_dir {OUTPUT_DIR}
"""

print("🚀 Starting training with paper-based configuration...\n")

# NOTE: This may take a long time to run
os.system(training_command)

print("\n" + "="*80)
print("✓ TRAINING COMPLETED!")
print("="*80)


FASE 4: START TRAINING - PAPER-BASED CONFIGURATION

📋 Training configuration summary:
   Model config: res18_paper_based.yaml
   Data config: data_colab_paper.yaml (vegetation-only)
   Trainer config: trainer_colab_paper.yaml

   🔴 PAPER-BASED MODIFICATIONS:
   ✓ Encoder weights: ImageNet (better feature recognition)
   ✓ Loss function: Focal Loss (handles class imbalance)
   ✓ Features: Vegetation only (VIIRS + NDVI + EVI2 + Fire)
   ✓ Batch size: 64 (exactly as in paper)
   ✓ Max steps: 10,000 (exactly as in paper)
   ✓ Checkpoint monitor: val_AP (DURANTE training)
   ✓ Training samples: 3948
   ✓ Validation samples: 3287

⚡ TRAINING PHASES:
   1. Training phase: aggiorna pesi basato su train loss
   2. Validation phase (ogni epoca): calcola val_AP, salva checkpoint se migliore
   3. Test phase (dopo training): calcola test_AP come metrica finale

🚀 Starting training with paper-based configuration...


✓ TRAINING COMPLETED!


In [ ]:
# ============================================================================
# Cell 9: MAIN TRAINING WORKLOAD - LUNCH PIPELINE
# ============================================================================

print("\n" + "="*80)
print("🚀 LAUNCHING EARTH TRAINING FACILITY - WORKLOAD: 10,000 STEPS")
print("="*80 + "\n")

print("📋 LOGISTICA DI ADDESTRAMENTO:")
print(f"  ├─ Modello:  U-Net ResNet-18 (ImageNet Weights Hardcoded)")
print(f"  ├─ Loss:     Focal Loss (Alpha Normalizzato: self.pos_class_weight / (1 + self.pos_class_weight))")
print(f"  ├─ Dati:     Vegetation-Only (VIIRS + NDVI + EVI2 + Active Fire)")
print(f"  ├─ Batch:    64 (Configurazione Lahrichi)")
print(f"  └─ Monitor:  val_AP (Validation Average Precision per checkpoint)")

print("\n📡 Connessione a Weights & Biases in corso... Logs in tempo reale attivi.\n")

# Usiamo lo script multilinea racchiuso nel comando magico "!" di Jupyter.
# Abbiamo impostato --do_validate true per garantire il funzionamento del monitor val_AP.
!cd /content/WildfireSpreadTS && \
python src/train.py \
  -c cfgs/unet/res18_paper_based.yaml \
  --trainer cfgs/trainer_colab_paper.yaml \
  --data cfgs/data_colab_paper.yaml \
  --do_train true \
  --do_test false \
  --do_validate true \
  --trainer.default_root_dir /content/lightning_logs

print("\n" + "="*80)
print("🎉 ENGINE FINISHED: TRAINING WORKLOAD COMPLETATO CON SUCCESSO!")
print("="*80)


🚀 LAUNCHING EARTH TRAINING FACILITY - WORKLOAD: 10,000 STEPS

📋 LOGISTICA DI ADDESTRAMENTO:
  ├─ Modello:  U-Net ResNet-18 (ImageNet Weights Hardcoded)
  ├─ Loss:     Focal Loss (Alpha Normalizzato: self.pos_class_weight / (1 + self.pos_class_weight))
  ├─ Dati:     Vegetation-Only (VIIRS + NDVI + EVI2 + Active Fire)
  ├─ Batch:    64 (Configurazione Lahrichi)
  └─ Monitor:  val_AP (Validation Average Precision per checkpoint)

📡 Connessione a Weights & Biases in corso... Logs in tempo reale attivi.

Seed set to 0
Using the following dataset split:
Train years: [2018, 2019], Val years: [2020], Test years: [2021]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
wandb: WARNING The anonymous setting has no effect and will be remove

### 10. Load the Best Checkpoint
We locate the best performing model checkpoint inside the `lightning_logs` directory based on the `val_AP` monitoring.


In [15]:
checkpoint_dir = Path(OUTPUT_DIR)
checkpoints = list(checkpoint_dir.glob("**/checkpoints/*.ckpt"))

if checkpoints:
    best_checkpoint = sorted(checkpoints, key=lambda x: x.stat().st_mtime)[-1]
    print("\n✓ Best checkpoint:")
    print(f"  Path: {best_checkpoint}")
    print(f"  Size: {best_checkpoint.stat().st_size / 1e6:.2f} MB")
    print("  Reason: Selected by val_AP (validation metric)")
else:
    print("\n✗ No checkpoints found")


✗ No checkpoints found


### 11. Testing Phase (Evaluation)
We evaluate the finalized model using the test split. The core metric here is **`test_AP`**.


In [ ]:
if checkpoints:
    print("\n" + "=" * 80)
    print("STARTING MODEL EVALUATION (TEST SET)...")
    print("=" * 80)

    print(f"\n📊 Using checkpoint: {best_checkpoint.name}")
    print(f"   Size: {best_checkpoint.stat().st_size / 1e6:.2f} MB")
    print("   Configuration:")
    print("   - Model: res18_paper_based.yaml")
    print("   - Loss: Focal Loss")
    print("   - Weights: ImageNet")
    print("   - Data: vegetation-only features")
    print(f"   - Test samples: {len(datamodule.test_dataset)}\n")

    print("⚡ TEST PHASE:")
    print("   - Calcola test_AP (metrica finale su dati non visti)")
    print("   - Calcola test_F1, test_precision, test_recall, test_iou")
    print("   - Genera confusion matrix e PR curve\n")

    test_command = f"""
    cd /content/WildfireSpreadTS && \
    python src/train.py \
      -c cfgs/unet/res18_paper_based.yaml \
      --trainer cfgs/trainer_colab_paper.yaml \
      --data cfgs/data_colab_paper.yaml \
      --do_train false \
      --do_test true \
      --do_validate false \
      --ckpt_path {str(best_checkpoint)} \
      --trainer.default_root_dir {OUTPUT_DIR}
    """

    print("🚀 Executing test command...\n")
    os.system(test_command)

    print("\n" + "=" * 80)
    print("✓ TEST COMPLETED!")
    print("=" * 80)
    print("\n📊 Metrics saved to:")
    print("   - Weights & Biases: https://wandb.ai/[username]/wildfire_resnet18_paper_based")
    print(f"   - Local logs: {OUTPUT_DIR}")
    print("\n📈 Test metrics (paper-based):")
    print("   - test_AP ← PRIMARY METRIC (metrica finale di valutazione)")
    print("   - test_f1, test_precision, test_recall")
    print("   - test_iou, confusion matrix, PR curve")
else:
    print("\n✗ Cannot run tests: no checkpoint found.")

### 12. Backup to Google Drive
Exporting logs and trained weights to ensure they persist beyond the Colab runtime lifetime.


In [ ]:
print("\n" + "="*80)
print("💾 SALVATAGGIO DI SICUREZZA SU GOOGLE DRIVE")
print("="*80 + "\n")

import shutil

drive_output = "/content/drive/MyDrive/Wildfire_Project/training_results_paper_based"
Path(drive_output).mkdir(parents=True, exist_ok=True)

try:
    if checkpoints:
        checkpoint_dest = f"{drive_output}/checkpoints"
        Path(checkpoint_dest).mkdir(parents=True, exist_ok=True)
        for ckpt in checkpoints:
            shutil.copy2(ckpt, f"{checkpoint_dest}/{ckpt.name}")
        print(f"✓ Checkpoints salvati in Drive: {checkpoint_dest}")
        print(f"  - File: {best_checkpoint.name}")
        print(f"  - Size: {best_checkpoint.stat().st_size / 1e6:.2f} MB")

    logs_dest = f"{drive_output}/lightning_logs"
    shutil.copytree(OUTPUT_DIR, logs_dest, dirs_exist_ok=True)
    print(f"\n✓ Log e metriche storiche salvati in: {logs_dest}")

    saved_files = len(list(Path(logs_dest).glob("**/*")))
    print(f"  - Total files: {saved_files}")

except Exception as e:
    print(f"⚠️ Errore durante il backup su Drive: {e}")

print("\n✓ Backup completato con successo!")

### 13. Compile ONNX Graph for Spatial Inference Workload
We perform a conversion of our `.ckpt` to the Open Neural Network Exchange format (ONNX), preparing it for lightweight, real-time edge computing on the target satellite.
*Note: we load explicitly the `SMPModel` with our specific geometric parameters instead of the abstract `BaseModel`.*


In [ ]:
print("\n" + "="*80)
print("🛰️ COMPILAZIONE GRAFO ONNX PER INFERENCE WORKLOAD IN ORBITA")
print("="*80 + "\n")

print("Fase: Conversione modello PyTorch → ONNX (64×64 pixel)")
print("Obiettivo: Modello leggero per inferenza real-time su satellite\n")

if checkpoints:
    try:
        # 🔴 FIX CRITICO: Non usare BaseModel.load_from_checkpoint()!
        # BaseModel è astratta e non ha i parametri geometrici.
        # Usa direttamente SMPModel con i parametri espliciti.

        print("1️⃣ Caricamento modello dal checkpoint...\n")

        print("   ⚠️  IMPORTANT: Loading SMPModel with explicit parameters")
        print(f"      - n_channels: {actual_n_channels}")
        print("      - encoder_name: resnet18")
        print("      - encoder_weights: imagenet (già addestrato)\n")

        # 🟢 CORRETTO: Usa SMPModel con parametri espliciti
        model_onnx = SMPModel.load_from_checkpoint(
            str(best_checkpoint),
            encoder_name="resnet18",
            n_channels=actual_n_channels,
            flatten_temporal_dimension=True,
            pos_class_weight=236,
            loss_function="Focal",
            # encoder_weights="imagenet"
        )

        print("   ✓ SMPModel caricato correttamente dal checkpoint")
        print(f"   ✓ Canali ricostruiti: {actual_n_channels}")

        model_onnx.eval()
        model_onnx.to('cpu')
        print("   ✓ Modello in eval mode e spostato su CPU\n")

        # ============================================================================
        # Preparazione input dummy per ONNX export
        # ============================================================================

        print("2️⃣ Preparazione input dummy per ONNX export...\n")

        # The inference on satellite will operate on 64x64 patches
        dummy_satellite_input = torch.randn(1, actual_n_channels, 64, 64)
        print(f"   - Input shape: {dummy_satellite_input.shape}")
        print("   - Risoluzione satellite: 64×64 pixel")
        print(f"   - Canali: {actual_n_channels}")
        print("   - Batch size: 1 (singola tile)\n")

        print("3️⃣ Verificazione forward pass prima di ONNX export...")
        with torch.no_grad():
            test_output = model_onnx(dummy_satellite_input)
        print("   ✓ Forward pass successful!")
        print(f"   - Output shape: {test_output.shape}")
        print(f"   - Output dtype: {test_output.dtype}\n")

        # ============================================================================
        # Esportazione ONNX
        # ============================================================================

        onnx_dest_path = "/content/drive/MyDrive/Wildfire_Project/wsts_paper_model.onnx"

        print("4️⃣ Esportazione a formato ONNX...\n")

        torch.onnx.export(
            model_onnx,
            dummy_satellite_input,
            onnx_dest_path,
            export_params=True,           # Salva i pesi addestrati
            opset_version=14,             # ONNX opset 14 (compatibile)
            do_constant_folding=True,     # Ottimizzazione costanti
            input_names=['input'],
            output_names=['output'],
            dynamic_axes={
                'input': {0: 'batch_size'},
                'output': {0: 'batch_size'}
            },
            verbose=False
        )

        onnx_file_size = Path(onnx_dest_path).stat().st_size / 1e6
        print("   ✓ ONNX export completato!\n")

    except Exception as e:
        print(f"❌ Errore critico durante caricamento o esportazione ONNX: {e}")
        import traceback
        traceback.print_exc()
else:
    print("❌ Esportazione fallita: nessun checkpoint rilevato.")

### 14. Validate the exported ONNX model
Run a final validation loop using `onnxruntime` to ensure that the exported file is operational and mathematically matches our expectations.


In [ ]:
if checkpoints and Path(onnx_dest_path).exists():
    try:
        # ============================================================================
        # Statistiche e validazione
        # ============================================================================

        print("5️⃣ Statistiche modello ONNX:")
        print(f"   - Path: {onnx_dest_path}")
        print(f"   - Size: {onnx_file_size:.2f} MB")
        print("   - Opset version: 14")
        print("   - Status: ✅ Pronto per deployment su satellite\n")

        print("6️⃣ Validazione ONNX Runtime...\n")
        import onnxruntime as ort

        # Carica il modello ONNX con ONNX Runtime
        sess = ort.InferenceSession(onnx_dest_path, providers=['CPUExecutionProvider'])
        print("   ✓ ONNX Runtime session creata")

        # Ispeziona il grafo ONNX
        input_name = sess.get_inputs()[0].name
        output_name = sess.get_outputs()[0].name
        input_shape = sess.get_inputs()[0].shape

        print(f"   - Input node: '{input_name}'")
        print(f"   - Input shape: {input_shape}")
        print(f"   - Output node: '{output_name}'")
        print(f"   - Output shape: {sess.get_outputs()[0].shape}\n")

        # Test inferenza con ONNX Runtime
        print("7️⃣ Test inferenza ONNX Runtime...\n")
        ort_input = dummy_satellite_input.cpu().numpy().astype('float32')
        ort_output = sess.run(None, {input_name: ort_input})

        print("   ✓ Inferenza ONNX riuscita!")
        print(f"   - Input: {ort_input.shape}")
        print(f"   - Output: {ort_output[0].shape}")
        print(f"   - Output dtype: {ort_output[0].dtype}\n")

        print("="*80)
        print("🎉 MISSION COMPLETE: Modello ONNX spaziale snellito esportato!")
        print("="*80)

        print("\n📍 DEPLOYMENT INSTRUCTIONS:")
        print(f"   1. Download modello: {onnx_dest_path}")
        print("   2. Upload su satellite")
        print("   3. Installa ONNX Runtime (Python/C++): pip install onnxruntime")
        print("   4. Carica e inferisci:\n")
        print("      import onnxruntime as ort")
        print("      sess = ort.InferenceSession('wsts_paper_model.onnx')")
        print("      output = sess.run(None, {'input': tile_64x64})\n")
        print("   5. Processa output: (1, 1, 64, 64) → fire probability map")

    except Exception as e:
        print(f"   ⚠️ Errore ONNX Runtime: {e}")
        print("   (Il modello è comunque esportato, possibile problema di validazione)\n")

### 15. Final Report Summary
Quick overview of the pipeline's execution and recommendations for the subsequent steps.


In [ ]:
print("\n" + "="*80)
print("📊 FINAL REPORT - TRAINING COMPLETO")
print("="*80 + "\n")

print("✅ WORKFLOW COMPLETATO CON SUCCESSO!\n")

print("📋 FASI ESEGUITE:")
print("   1. ✓ Setup ambiente Colab + Google Drive")
print("   2. ✓ Preparazione dati (5 giorni, vegetazione-only)")
print("   3. ✓ Configurazione modello paper-based")
print("   4. ✓ Training (10,000 steps)")
print("   5. ✓ Testing (valutazione completa)")
print("   6. ✓ Backup su Google Drive")
print("   7. ✓ Export ONNX per deployment satellite\n")

print("🔴 MODIFICHE PAPER-BASED APPLICATE:")
print("   ✓ ImageNet pre-training (encoder)")
print("   ✓ Focal Loss (class imbalance)")
print("   ✓ Vegetation-only features (7 canali)")
print("   ✓ Batch size: 64")
print("   ✓ val_AP monitoring (checkpoint save DURANTE training)")
print("   ✓ test_AP evaluation (DOPO training)\n")

print("📈 METRICHE ATTESE:")
print("   • test_AP > 0.80 (target paper-based)")
print("   • test_F1 > 0.75")
print("   • test_precision > 0.80")
print("   • test_recall > 0.70\n")

print("💾 ARTIFACTS SALVATI:")
print(f"   • Checkpoint: {drive_output}/checkpoints/")
print(f"   • Logs: {drive_output}/lightning_logs/")
print("   • ONNX model: /content/drive/MyDrive/Wildfire_Project/wsts_paper_model.onnx\n")

print("🚀 PROSSIMI PASSI:")
print("   1. Monitorare metriche su W&B dashboard")
print("   2. Comparare test_AP con baseline (1 giorno)")
print("   3. Download checkpoint e ONNX da Drive")
print("   4. Deploy ONNX su satellite (ONNX Runtime)")
print("   5. Real-time inference su fire hotspots\n")

print("="*80)
print("🎊 TRAINING PIPELINE COMPLETATO CON SUCCESSO! 🎊")
print("="*80)